In [ ]:
# Day 17 - LoRA Fine-tuning (PEFT)
!pip install -q peft accelerate bitsandbytes trl datasets

import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer

In [ ]:
# Load Small Model in 4-bit
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    load_in_4bit=True
)

print("Base model loaded!")

In [ ]:
# LoRA Configuration
lora_config = LoraConfig(
    r=16,                    # Rank
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Create Small Custom Dataset (You can expand this)
data = [
    {"text": "Q: What is the capital of Ethiopia?\nA: The capital of Ethiopia is Addis Ababa."},
    {"text": "Q: What is the national dish of Ethiopia?\nA: The national dish is Injera with various stews."},
    {"text": "Q: Where was coffee discovered?\nA: Coffee was discovered in Ethiopia."},
]

from datasets import Dataset
dataset = Dataset.from_list(data)

def tokenize(example):
    return tokenizer(example["text"], truncation=True, max_length=256)

tokenized_dataset = dataset.map(tokenize, batched=True)

In [ ]:
# Training Arguments
training_args = TrainingArguments(
    output_dir="lora_finetuned_model",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    save_steps=50,
    logging_steps=10,
    learning_rate=2e-4,
    fp16=True,
    optim="adamw_8bit"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

trainer.train()

In [ ]:
# Save LoRA Adapter
model.save_pretrained("ethiopian_lora_adapter")
tokenizer.save_pretrained("ethiopian_lora_adapter")

print("✅ Fine-tuning completed and adapter saved!")